# Semana 16 · ¿Qué tira de la Luna?

Este cuaderno acompaña a la sesión de proyecto de la semana 16: vectores con NumPy y la ley de la gravitación universal.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Vectores con NumPy

Un vector se guarda como un `np.array` con sus componentes. NumPy suma, resta y multiplica componente a componente, que es justo lo que necesitamos. El módulo se calcula con `np.linalg.norm`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([1.0, 2.0])
B = np.array([4.0, 6.0])

AB = B - A                       # vector de A a B: "a dónde llego menos de dónde salgo"
modulo = np.linalg.norm(AB)
unitario = AB / modulo
print("Vector AB:", AB, "  módulo:", modulo, "  unitario:", unitario)
print("Módulo del unitario:", np.linalg.norm(unitario))

**Ejercicio.** Suma las fuerzas $(10, 0)$ N y una de 10 N a 120° del eje $x$. Calcula el módulo y el ángulo de la fuerza neta (con `np.degrees(np.arctan2(Fy, Fx))`).

In [ ]:
# Tu código aquí

In [ ]:
# Solución
F1 = np.array([10.0, 0.0])
ang = np.radians(120)
F2 = 10 * np.array([np.cos(ang), np.sin(ang)])
F = F1 + F2
print("Fuerza neta:", F.round(3), " módulo:", round(np.linalg.norm(F), 3), "N  ángulo:", round(np.degrees(np.arctan2(F[1], F[0])), 1), "°")

## 2. La fuerza gravitatoria como vector

La fuerza que el cuerpo 2 ejerce sobre el cuerpo 1 tiene módulo $G m_1 m_2 / r^2$ y apunta del cuerpo 1 hacia el cuerpo 2. Con el vector $\vec{d} = \vec{r}_2 - \vec{r}_1$, de módulo $r$, y su unitario $\hat{d} = \vec{d}/r$:

$$\vec{F}_{12} = G\,\frac{m_1 m_2}{r^2}\,\hat{d} = G\,\frac{m_1 m_2}{r^3}\,\vec{d}.$$

La segunda forma es la que se programa: evita calcular el unitario aparte.

In [ ]:
G = 6.674e-11     # N·m²/kg²

def fuerza_grav(m1, r1, m2, r2):
    """Fuerza (vector, en N) que el cuerpo 2 ejerce sobre el cuerpo 1. Posiciones en m, masas en kg."""
    d = r2 - r1
    r = np.linalg.norm(d)
    return G * m1 * m2 / r**3 * d

# Dos masas de 1 kg a 1 m
F = fuerza_grav(1.0, np.array([0.0, 0.0]), 1.0, np.array([1.0, 0.0]))
print("Fuerza entre dos masas de 1 kg a 1 m:", F, "N")

## 3. La gravedad en la superficie de los planetas

Con $g = GM/R^2$ y las masas y radios (ecuatoriales) de la hoja de datos de la NASA, redondeados.

In [ ]:
planetas = {            # masa (kg), radio (m)
    "Mercurio": (3.30e23, 2.440e6),
    "Venus":    (4.87e24, 6.052e6),
    "Tierra":   (5.97e24, 6.378e6),
    "Luna":     (7.35e22, 1.737e6),
    "Marte":    (6.42e23, 3.396e6),
    "Júpiter":  (1.898e27, 7.149e7),
}
for nombre, (M, R) in planetas.items():
    g = G * M / R**2
    print(f"{nombre:9s} g = {g:6.2f} m/s²   ({g / 9.81:.2f} veces la de la Tierra)")

En Júpiter no hay superficie sólida: el valor es el de la altura donde la presión es la de la superficie terrestre, que es la que se usa para definir su radio.

## 4. El Sol y la Tierra tirando de la Luna

Colocamos el Sol en el origen, la Tierra a 1 ua sobre el eje $x$ y la Luna a 384 400 km de la Tierra, en un ángulo $\phi$ que vamos variando. Calculamos las dos fuerzas sobre la Luna y su suma.

In [ ]:
ua = 1.496e11
M_S, M_T, M_L = 1.989e30, 5.972e24, 7.35e22
d_TL = 3.844e8

r_S = np.array([0.0, 0.0])
r_T = np.array([ua, 0.0])

print(" ángulo |  F Sol (10^20 N) | F Tierra (10^20 N) | F neta (10^20 N)")
for phi in [0, 90, 180, 270]:
    p = np.radians(phi)
    r_L = r_T + d_TL * np.array([np.cos(p), np.sin(p)])
    F_S = fuerza_grav(M_L, r_L, M_S, r_S)
    F_T = fuerza_grav(M_L, r_L, M_T, r_T)
    print(f"  {phi:4d}° | {np.linalg.norm(F_S)/1e20:15.3f}  | {np.linalg.norm(F_T)/1e20:17.3f}  | {np.linalg.norm(F_S + F_T)/1e20:14.3f}")

El Sol tira de la Luna más del doble que la Tierra, en cualquier posición. La fuerza neta sobre la Luna apunta casi siempre hacia el Sol. ¿Por qué no se separa la Luna de la Tierra? Porque el Sol tira **casi lo mismo** de la Tierra por unidad de masa: las dos caen juntas alrededor del Sol. Lo comprobamos comparando las **aceleraciones** que produce el Sol en cada una (fuerza dividida por masa).

In [ ]:
for phi in [0, 90, 180]:
    p = np.radians(phi)
    r_L = r_T + d_TL * np.array([np.cos(p), np.sin(p)])
    a_L = fuerza_grav(M_L, r_L, M_S, r_S) / M_L
    a_T = fuerza_grav(M_T, r_T, M_S, r_S) / M_T
    dif = a_L - a_T
    print(f"{phi:3d}°  aceleración del Sol sobre la Luna: {np.linalg.norm(a_L):.6f} m/s²   sobre la Tierra: {np.linalg.norm(a_T):.6f} m/s²   diferencia: {np.linalg.norm(dif):.2e} m/s²")

a_tierra_sobre_luna = G * M_T / d_TL**2
print(f"\nAceleración que produce la Tierra sobre la Luna: {a_tierra_sobre_luna:.6f} m/s²")

La diferencia entre lo que el Sol acelera a la Luna y a la Tierra es unas cien veces menor que la aceleración que la Tierra produce en la Luna. Para el movimiento de la Luna **alrededor de la Tierra**, el Sol es solo una pequeña perturbación. Esa diferencia de aceleraciones es la **fuerza de marea** del Sol, y la estudiarás en la semana 21.

## 5. La prueba de la Luna de Newton

**Ejercicio.** Calcula dos aceleraciones y compáralas:

1. la de la gravedad terrestre a la distancia de la Luna, suponiendo que disminuye con el cuadrado de la distancia: $g\,(R_T/r)^2$, con $g = 9{,}81$ m/s² y $R_T = 6371$ km;
2. la que necesita la Luna para su órbita circular: $4\pi^2 r/T^2$, con $T = 27{,}32$ días.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
g, R_T, r = 9.81, 6.371e6, 3.844e8
T = 27.32 * 86400
a_gravedad = g * (R_T / r)**2
a_orbita = 4 * np.pi**2 * r / T**2
print(f"Gravedad terrestre a la distancia de la Luna: {a_gravedad:.6f} m/s²")
print(f"Aceleración que necesita la órbita de la Luna: {a_orbita:.6f} m/s²")
print(f"Diferencia: {100 * (a_orbita - a_gravedad) / a_gravedad:.1f} %")

## 6. Experimenta

1. Dibuja la gravedad terrestre en función de la altura sobre la superficie, desde 0 hasta 40 000 km. ¿A qué altura vale la mitad que en la superficie? ¿Cuánto vale a la altura de la Estación Espacial Internacional, unos 400 km?
2. ¿Dónde, en la línea entre la Tierra y la Luna, se anulan las dos atracciones sobre una nave? Pista: busca el punto donde $GM_T/x^2 = GM_L/(d - x)^2$, probando valores de $x$ o despejando con una raíz cuadrada.

In [ ]:
# Tu experimento aquí